# Adquisición y normalización con pandas

**Jorge Navarrete Villanueva · Data Science II**

Integro las transacciones del área de ventas con el catálogo de productos para calcular el importe de cada venta. Los datos son ficticios. Cada fila final representa una transacción validada y los importes se expresan en soles (PEN).

## 1. Adquisición de las fuentes

El CSV contiene las transacciones. El Excel contiene un registro por producto. Uso `ExcelFile` como gestor de contexto para cerrar el archivo después de leer la hoja del catálogo.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

carpeta_datos = Path("datos")
ruta_parquet = Path("salida") / "ventas_normalizadas.parquet"

ventas = pd.read_csv(carpeta_datos / "transacciones.csv", dtype="string")
with pd.ExcelFile(carpeta_datos / "catalogo_productos.xlsx", engine="openpyxl") as archivo:
    catalogo = pd.read_excel(archivo, sheet_name="catalogo")

print(f"Transacciones de origen: {len(ventas)}")
print(f"Productos en el catálogo: {len(catalogo)}")
print(f"Tipo inicial del ID en el CSV: {ventas['id_producto'].dtype}")
print(f"Tipo inicial del ID en el Excel: {catalogo['id_producto'].dtype}")
display(ventas.head(3), catalogo.head(3))

Transacciones de origen: 30
Productos en el catálogo: 7
Tipo inicial del ID en el CSV: string
Tipo inicial del ID en el Excel: int64


,id_transaccion,id_producto,cantidad,fecha
0,V001,00101,2,2026-10-01
1,V002,102,3,2026-10-01
2,V003,103,1,2026-10-02


,id_producto,nombre_producto,precio_unitario
0,101,Teclado,79.9
1,102,Mouse,34.5
2,103,Audífonos,119.0


## 2. Tipos de datos y registros incompletos

Los ID de producto y las cantidades válidas son enteros positivos. Convierto las fechas ISO a `datetime` y reviso que cada venta tenga un identificador único. Si faltan ID, cantidad o fecha, excluyo la venta y registro el motivo. Inventar esos datos cambiaría el producto, la cantidad o el momento de la transacción.

In [2]:
def entero_positivo(serie):
    numeros = pd.to_numeric(serie, errors="coerce")
    return numeros.where(numeros.gt(0) & numeros.mod(1).eq(0)).astype("Int64")

ventas["id_transaccion"] = ventas["id_transaccion"].str.strip().replace("", pd.NA)
ventas["id_producto"] = entero_positivo(ventas["id_producto"])
ventas["cantidad"] = entero_positivo(ventas["cantidad"])
ventas["fecha"] = pd.to_datetime(ventas["fecha"], format="%Y-%m-%d", errors="coerce")

catalogo["id_producto"] = entero_positivo(catalogo["id_producto"])
catalogo["nombre_producto"] = catalogo["nombre_producto"].astype("string").str.strip().replace("", pd.NA)
precios = pd.to_numeric(catalogo["precio_unitario"], errors="coerce")
catalogo["precio_unitario"] = precios.where(precios.ge(0) & precios.lt(float("inf")))

if ventas["id_transaccion"].isna().any() or ventas["id_transaccion"].duplicated().any():
    raise ValueError("Los ID de transacción deben estar completos y ser únicos.")
if catalogo["id_producto"].isna().any():
    raise ValueError("El catálogo contiene un ID de producto vacío o inválido.")

reglas = {
    "id_producto": "ID de producto vacío o inválido",
    "cantidad": "Cantidad vacía o inválida",
    "fecha": "Fecha vacía o inválida",
}
motivo = pd.Series(pd.NA, index=ventas.index, dtype="string")
for columna, descripcion in reglas.items():
    motivo.loc[motivo.isna() & ventas[columna].isna()] = descripcion

descartadas_origen = ventas.loc[motivo.notna(), ["id_transaccion", "id_producto"]].copy()
descartadas_origen["motivo"] = motivo.loc[motivo.notna()]
ventas_validas = ventas.loc[motivo.isna()].copy()

print(f"Tipo del ID en ambos orígenes: {ventas['id_producto'].dtype} / {catalogo['id_producto'].dtype}")
print(f"Tipo de la fecha: {ventas_validas['fecha'].dtype}")
print(f"Transacciones aptas para el cruce: {len(ventas_validas)}")

Tipo del ID en ambos orígenes: Int64 / Int64
Tipo de la fecha: datetime64[ns]
Transacciones aptas para el cruce: 26


## 3. Integración por ID de producto

Un producto puede aparecer en varias ventas y debe tener una sola ficha en el catálogo. `validate="many_to_one"` comprueba esa relación. La unión izquierda permite detectar qué ventas quedaron sin coincidencia. Si falta la ficha, el nombre o el precio, separo la transacción para que el importe consolidado se calcule con datos comprobables.

In [3]:
cruce = ventas_validas.merge(
    catalogo,
    on="id_producto",
    how="left",
    validate="many_to_one",
    indicator=True,
)

sin_producto = cruce["_merge"].eq("left_only")
catalogo_incompleto = cruce["_merge"].eq("both") & cruce[
    ["nombre_producto", "precio_unitario"]
].isna().any(axis=1)

columnas_auditoria = ["id_transaccion", "id_producto"]
descartadas_cruce = pd.concat([
    cruce.loc[sin_producto, columnas_auditoria].assign(motivo="Producto sin coincidencia en el catálogo"),
    cruce.loc[catalogo_incompleto, columnas_auditoria].assign(motivo="Ficha de catálogo incompleta"),
], ignore_index=True)
descartadas = pd.concat([descartadas_origen, descartadas_cruce], ignore_index=True)
descartadas = descartadas.sort_values("id_transaccion").reset_index(drop=True)

consolidadas = cruce.loc[~(sin_producto | catalogo_incompleto)].drop(columns="_merge").copy()
print(f"Filas antes y después de unir: {len(ventas_validas)} -> {len(cruce)}")
print(f"Transacciones excluidas: {len(descartadas)}")
display(descartadas)

Filas antes y después de unir: 26 -> 26
Transacciones excluidas: 6


,id_transaccion,id_producto,motivo
0,V025,101,Cantidad vacía o inválida
1,V026,102,Fecha vacía o inválida
2,V027,<NA>,ID de producto vacío o inválido
3,V028,999,Producto sin coincidencia en el catálogo
4,V029,107,Ficha de catálogo incompleta
5,V030,104,Cantidad vacía o inválida


## 4. Importe y controles de calidad

`total_venta` es `cantidad × precio_unitario`, redondeado a dos decimales por transacción. Verifico que el cruce conserve el número de filas esperado, que todas las transacciones se puedan conciliar y que las columnas de salida estén completas.

In [4]:
consolidadas["total_venta"] = (consolidadas["cantidad"] * consolidadas["precio_unitario"]).round(2)
columnas_finales = [
    "id_transaccion", "id_producto", "nombre_producto", "cantidad",
    "fecha", "precio_unitario", "total_venta",
]
consolidadas = consolidadas[columnas_finales].sort_values(["fecha", "id_transaccion"]).reset_index(drop=True)

assert len(cruce) == len(ventas_validas), "El cruce cambió la cantidad de transacciones aptas."
assert len(ventas) == len(consolidadas) + len(descartadas), "Hay filas sin conciliar."
assert not consolidadas.empty, "No quedaron transacciones válidas."
assert consolidadas["id_transaccion"].is_unique, "Hay transacciones repetidas."
assert pd.api.types.is_datetime64_any_dtype(consolidadas["fecha"]), "La fecha debe ser datetime."
assert not consolidadas.isna().any().any(), "El resultado contiene valores nulos."

print(f"Entradas: {len(ventas)} | Válidas: {len(consolidadas)} | Excluidas: {len(descartadas)}")
print(f"Unidades vendidas: {consolidadas['cantidad'].sum()}")
print(f"Importe total en PEN: {consolidadas['total_venta'].sum():.2f}")
display(consolidadas.head())

Entradas: 30 | Válidas: 24 | Excluidas: 6
Unidades vendidas: 58
Importe total en PEN: 3565.90


,id_transaccion,id_producto,nombre_producto,cantidad,fecha,precio_unitario,total_venta
0,V001,101,Teclado,2,2026-10-01,79.9,159.8
1,V002,102,Mouse,3,2026-10-01,34.5,103.5
2,V003,103,Audífonos,1,2026-10-02,119.0,119.0
3,V004,104,Webcam,2,2026-10-02,145.0,290.0
4,V005,105,Hub USB,3,2026-10-03,65.9,197.7


## 5. Exportación y lectura de control

Guardo el consolidado en Parquet sin incluir el índice del notebook. Vuelvo a leerlo y lo comparo con el resultado en memoria para comprobar sus valores y tipos.

In [5]:
ruta_parquet.parent.mkdir(parents=True, exist_ok=True)
consolidadas.to_parquet(ruta_parquet, engine="pyarrow", index=False)
recuperadas = pd.read_parquet(ruta_parquet, engine="pyarrow")
pd.testing.assert_frame_equal(consolidadas, recuperadas)

print(f"Parquet verificado: {ruta_parquet.as_posix()}")
print(f"Filas: {len(recuperadas)} | Nulos: {recuperadas.isna().sum().sum()}")

Parquet verificado: salida/ventas_normalizadas.parquet
Filas: 24 | Nulos: 0
